## 01 - Capture BLE IQ Bursts and BLE Scan Logs

This notebook captures BLE advertising bursts using PlutoSDR and logs BLE MAC addresses using the Raspberry Pi's Bluetooth stack. The configuration below controls all parameters explicitly.


In [ ]:
import numpy as np
import time
import os  # <- ADD THIS
from bluepy.btle import Scanner
import adi  # PlutoSDR driver
import sys
import threading

# Add the src folder to Python path
sys.path.append(os.path.abspath("../src"))
from utils import current_utc_timestamp, ensure_dir
# Initialize capture tools
from capture import PlutoSDRCapture, BLEScanner


# Example config
config = {
    'center_freq': 2402e6,          # BLE channel 37
    'sample_rate': 2500000,
    'num_samples': 1024,
    'threshold': 0.01,
    'duration': 60,  # seconds
    'save_path': 'data/raw',
    'ble_log_path': 'data/ble_log.csv',
    'ble_scan_interval': 5,  # seconds
}

In [ ]:
!which python3

In [ ]:
import numpy as np
import time
import os
import sys
import threading
import subprocess

# Add the src folder to Python path
sys.path.append(os.path.abspath("../src"))

from utils import current_utc_timestamp, ensure_dir
from capture import PlutoSDRCapture

# ------------------------
# ? Configuration
# ------------------------
config = {
    'center_freq': 2402e6,          # BLE channel 37
    'sample_rate': 2500000,
    'num_samples': 1024,
    'threshold': 0.01,
    'duration': 60,  # seconds
    'save_path': '../data/raw',
    'ble_log_path': '../data/ble_log.csv',
    'ble_scan_interval': 5,  # seconds
}

ensure_dir(config['save_path'])
ensure_dir(os.path.dirname(config['ble_log_path']))

# ------------------------
# ? BLE subprocess launch
# ------------------------
ble_process = subprocess.Popen(
    [
        "sudo",
        "/home/isesdr/Downloads/ble-RFFingerprinting/ble-rff-env/bin/python",
        "../src/ble_scan.py",
        "--duration", str(config['duration']),
        "--log", config['ble_log_path'],
        "--interval", str(config['ble_scan_interval'])
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# ? Optional: live stream BLE output in background
def stream_output(process):
    for line in iter(process.stdout.readline, ""):
        print("[BLE]", line.strip())

ble_log_thread = threading.Thread(target=stream_output, args=(ble_process,), daemon=True)
ble_log_thread.start()

# ------------------------
# ? Wait for BLE ready
# ------------------------
print("?? Waiting for BLE scanner to initialize...")
while True:
    line = ble_process.stdout.readline()
    if not line:
        break
    print("[BLE]", line.strip())
    if "READY" in line:
        print("?? BLE scanner is running.")
        break

# ------------------------
# ? Start SDR capture
# ------------------------
sdr = PlutoSDRCapture(
    center_freq=config['center_freq'],
    sample_rate=config['sample_rate'],
    num_samples=config['num_samples'],
    threshold=config['threshold']
)

try:
    print(f"?? SDR capture start: {current_utc_timestamp()}")
    sdr.capture_bursts(duration=config['duration'], save_path=config['save_path'])
    print(f"?? SDR capture end: {current_utc_timestamp()}")

    print("?? Waiting for BLE scanner to finish...")
    ble_process.wait()
    print("? All capture complete.")

except KeyboardInterrupt:
    print("? Interrupted! Cleaning up...")
    sdr.stop()
    ble_process.terminate()
    try:
        ble_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        print("?? BLE scanner did not exit in time.")


In [ ]:
import numpy as np
import time
import os
import sys
import threading
import subprocess

sys.path.append(os.path.abspath("../src"))
from utils import current_utc_timestamp, ensure_dir
from capture import PlutoSDRCapture

# --- Configuration
config = {
    'center_freq': 2402e6,
    'sample_rate': 2500000,
    'num_samples': 1024,
    'threshold': 0.01,
    'duration': 3600,
    'save_path': '../data/raw',
    'ble_log_path': '../data/ble_log.csv',
    'ble_scan_interval': 1,
}

ensure_dir(config['save_path'])
ensure_dir(os.path.dirname(config['ble_log_path']))

# --- Launch BLE subprocess
ble_process = subprocess.Popen(
    [
        "sudo",
        "/home/isesdr/Downloads/ble-RFFingerprinting/ble-rff-env/bin/python",
        "../src/ble_scan.py",
        "--duration", str(config['duration']),
        "--log", config['ble_log_path'],
        "--interval", str(config['ble_scan_interval'])
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# --- Wait for BLE scanner to be READY
def wait_for_ready(proc):
    for line in iter(proc.stdout.readline, ""):
        print("[BLE]", line.strip())
        if "READY" in line:
            break

wait_for_ready(ble_process)

# --- Start SDR in a thread
def run_sdr():
    sdr = PlutoSDRCapture(
        center_freq=config['center_freq'],
        sample_rate=config['sample_rate'],
        num_samples=config['num_samples'],
        threshold=config['threshold']
    )
    print(f"?? SDR capture start: {current_utc_timestamp()}")
    sdr.capture_bursts(duration=config['duration'], save_path=config['save_path'])
    print(f"? SDR capture end: {current_utc_timestamp()}")

sdr_thread = threading.Thread(target=run_sdr)
sdr_thread.start()

# --- Final sync
sdr_thread.join()
ble_process.wait()
print("? BLE + SDR capture complete.")

[BLE] READY
